<a href="https://colab.research.google.com/github/Timbo3399/INFM_SOTA2/blob/main/notebooks/workbook_timo_cluster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 6: Pre-Training auf dem Cluster

**Autor:** Timo · **Datum:** 21.11.2026

Wir trainieren GPT-2 auf dem Cluster DACHS, mit den Optimierungen aus Woche 5 und mit DDP über mehrere Knoten. Dieses Notebook trainiert nichts. Es liest die Ergebnisdateien, die die Jobs nach `results/week06/` schreiben, und läuft ohne GPU.

| Nr. | Lauf | Wozu |
|---|---|---|
| 1 | Benchmark | GPU und Treiber funktionieren, die L40S neben der A100, größte Batchgröße |
| 2 | Einstellungen aus Woche 4 | Rechnet der Cluster wie Colab? |
| 3 | Optimiert | Wie viel schneller wird das echte Training? |
| 4 | Suche | beste Kombination aus Lernrate und Batchgröße nach Validierungs-Loss |
| 5 | Skalierung | Speedup und Effizienz auf 1, 2 und 4 Knoten |
| 6 | Hauptlauf | Pre-Training von GPT-2 Small auf 2 Knoten |
| 7 | GPT-2 Medium | dasselbe mit dem größeren Modell |
| 8 | GPT-2 Large | optional |

Welche Läufe es gibt und warum, steht in [`docs/cluster-training.md`](https://github.com/Timbo3399/INFM_SOTA2/blob/main/docs/cluster-training.md), wie man sie startet, in [`docs/dachs.md`](https://github.com/Timbo3399/INFM_SOTA2/blob/main/docs/dachs.md). Die Tabellen rechnet `infm_sota2.cluster.results`, der Code hat Unittests.

## 0. Einstellungen und Setup

Mit `SMOKE_TEST = True` trainiert das Notebook ein paar winzige Läufe auf der CPU in einem temporären Ordner und zeigt damit jede Tabelle und jedes Diagramm. Die Zahlen haben dann keine Aussagekraft. Mit `False` liest es die Ergebnisse aus `results/week06/`. Fehlen dort Dateien, sagt die Zelle, welcher Lauf noch aussteht.

Lokal liest das Notebook den eigenen Klon, in Colab holt es das Repo vom Branch `BRANCH`. `PERSON` beschränkt die Auswertung auf die Läufe einer Person, so wie ihr Ordner unter `results/week06/` heißt. Mit `None` zählen alle. Haben zwei Personen dieselbe Kombination der Suche gerechnet, bricht die Tabelle mit beiden Dateinamen ab, dann hilft `PERSON`.

In [ ]:
SMOKE_TEST = False

In [ ]:
import sys
from pathlib import Path

# Colab installiert das Paket aus diesem Branch und holt die Ergebnisdateien von dort.
# Vor dem Merge nach main zum Testen vorübergehend z. B. "week/06" eintragen und vor
# dem Speichern zurück auf "main" setzen.
BRANCH = "main"
PERSON = None  # nur die Läufe dieser Person, z. B. "of_<HSO-Kürzel>"; None: alle
if "google.colab" in sys.modules:
    # Eigenes Paket installieren, Colabs torch/tiktoken/wandb bleiben unangetastet
    %pip install -q --no-deps "git+https://github.com/Timbo3399/INFM_SOTA2.git@{BRANCH}"
    REPO = Path("/content/INFM_SOTA2")
    if not REPO.exists():
        !git clone -q --depth 1 --branch {BRANCH} https://github.com/Timbo3399/INFM_SOTA2.git {REPO}
else:
    # Lokal: der Klon, in dem dieses Notebook liegt
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
print("Repo:", REPO)

In [ ]:
import io
import os
import tempfile
from contextlib import redirect_stdout
from dataclasses import replace

import pandas as pd

from infm_sota2.cluster.config import SearchSettings, load_config, with_grid
from infm_sota2.cluster.results import load_results

# Gitter der Suche wie in configs/lotr-small-search.toml, im Smoke-Test 2 × 2
SEARCH = load_config(REPO / "configs" / "lotr-small-search.toml").search
if SMOKE_TEST:
    SEARCH = SearchSettings(learning_rates=(1e-3, 1e-2), batch_sizes=(2, 4), epochs=1)


def smoke_results() -> Path:
    """Winzige echte Läufe auf der CPU, einer pro Abschnitt, in einem temporären Ordner."""
    from infm_sota2.cluster import benchmark
    from infm_sota2.cluster.train import run
    from infm_sota2.data.mini_book import write_mini_book
    from infm_sota2.env import Environment

    base = Path(tempfile.mkdtemp(prefix="sota2-cluster-smoke-"))
    env = replace(
        Environment.detect(smoke_test=True),
        base_dir=base,
        data_dir=base / "data",
    )
    env.data_dir.mkdir()
    write_mini_book(env.data_dir)
    smoke = replace(load_config(REPO / "configs" / "smoke-cpu.toml"), save_checkpoint=False)
    sections = [
        ("lotr-small-week4", "week6-week4"),
        ("lotr-small", "week6-optimized"),
        ("lotr-small-scaling", "week6-scaling"),
        ("lotr-small-main", "week6-main"),
        ("lotr-medium", "week6-medium"),
    ]
    configs = [replace(smoke, name=name, group=group) for name, group in sections]
    search = replace(smoke, name="lotr-small-search", group="week6-search", search=SEARCH)
    configs += [with_grid(search, i) for i in range(len(SEARCH.learning_rates) * len(SEARCH.batch_sizes))]
    working_dir = Path.cwd()
    with redirect_stdout(io.StringIO()):
        for config in configs:
            run(config, env, "smoke", None, REPO)
        os.chdir(base)  # der Benchmark schreibt unter den Arbeitsordner
        try:
            benchmark.main(["--smoke"])
        finally:
            os.chdir(working_dir)
    return base / "results" / "week06"


RESULTS = smoke_results() if SMOKE_TEST else REPO / "results" / "week06"


def owner(record: dict) -> str:
    """Die Person, unter deren Ordner die Datei liegt."""
    return Path(record["file"]).relative_to(RESULTS).parts[0]


RECORDS = load_results(RESULTS) if RESULTS.is_dir() else []
if PERSON is not None and not SMOKE_TEST:
    RECORDS = [record for record in RECORDS if owner(record) == PERSON]
TRAINING = [record for record in RECORDS if "group" in record]
BENCHMARKS = [record for record in RECORDS if "batch_scan" in record]


def by_group(group: str) -> list[dict]:
    return [record for record in TRAINING if record["group"] == group]


if SMOKE_TEST:
    print("SMOKE_TEST: winzige Läufe auf der CPU, Zahlen ohne Aussagekraft")
print(f"{len(TRAINING)} Trainingsläufe und {len(BENCHMARKS)} Benchmarks unter {RESULTS}")

In [ ]:
import matplotlib.pyplot as plt

# Nach dem Skill dataviz: Serienfarben in fester Reihenfolge (die ersten drei der
# geprüften Palette, blau, orange, aqua), Text in neutralen Tönen statt in der
# Serienfarbe, dünne Linien, zurückhaltendes Gitter, eine y-Achse pro Diagramm
INK, INK_SECONDARY, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE = "#e1e0d9", "#c3c2b7", "#fcfcfb"
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "text.color": INK,
        "axes.titlecolor": INK,
        "axes.labelcolor": INK_SECONDARY,
        "axes.edgecolor": AXIS,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.6,
        "xtick.color": AXIS,
        "ytick.color": AXIS,
        "xtick.labelcolor": INK_SECONDARY,
        "ytick.labelcolor": INK_SECONDARY,
        "lines.linewidth": 1.5,
        "lines.markersize": 6,
        "legend.frameon": False,
        "figure.dpi": 110,
    }
)


def summary(records: list[dict]) -> pd.DataFrame:
    """Eine Zeile pro Lauf. Läufe aus Schritt 2 haben noch kein mfu und keine Tokens/s."""
    rows = []
    for record in records:
        result = record["result"] or {}
        train = record["config"]["train"]
        rows.append(
            {
                "Lauf": record["label"],
                "Person": owner(record),
                "Modell": record["config"]["model_class"],
                "Präzision": train["precision"],
                "compile": train["compile_backend"] or "-",
                "GPUs": record["environment"]["world_size"],
                "Lernrate": train["learning_rate"],
                "Batch": train["batch_size"],
                "Test-Loss": record["test_loss"],
                "Perplexity": record["perplexity"],
                "bester Val-Loss": result.get("best_val_loss"),
                "Tokens/s": result.get("tokens_per_second"),
                "MFU": record.get("mfu"),
                "Speicher (GB)": record["peak_memory_gb"],
                "fit (s)": record["fit_seconds"],
                "Fehler": record["error"],
            }
        )
    return pd.DataFrame(rows).set_index("Lauf")


def show_texts(record: dict) -> None:
    print(f"{record['label']} ({owner(record)}):")
    for text, copied in zip(record["texts"] or [], record["copied"] or []):
        print("  " + " ".join(text.split()))  # Zeilenumbrüche der Texte zusammenfassen
        print(f"    längste kopierte Folge: {len(copied.split())} Wörter")

## 1. Job 1: Umgebung

`slurm/check.sbatch` prüft, ob alles da ist, bevor wir trainieren. Wichtig sind der Treiber (torch 2.14.1 bringt die CUDA-13.0-Laufzeit mit), der Test des Flash-Kernels, der C-Compiler für `torch.compile`, das Internet auf den Rechenknoten (sonst `WANDB_MODE=offline`) und der Datenordner. Die Tabelle zeigt die Umgebung, die jeder Benchmark mitgeschrieben hat.

In [ ]:
if not BENCHMARKS:
    print(f"Noch kein Ergebnis von Job 1 unter {RESULTS}. Start: sbatch slurm/check.sbatch")
else:
    display(pd.DataFrame({Path(record["file"]).name: record["environment"] for record in BENCHMARKS}))

## 2. Benchmark (Lauf Nr. 1)

Dieselben sechs Varianten wie in Woche 5: GPT-2 Small, Batch 4, 1024 Tokens, 3 Aufwärm- und 10 Messschritte. Die A100-Werte stammen aus [`old/docs/attention-v2.md`](https://github.com/Timbo3399/INFM_SOTA2/blob/main/old/docs/attention-v2.md). Danach misst der Job Variante 6 bei 256 Tokens wie im Training mit wachsender Batchgröße, bis der Speicher nicht mehr reicht. Die größte Batchgröße ohne Speicherfehler ist die Obergrenze für die Suche.

In [ ]:
# Woche 5, old/docs/attention-v2.md: NVIDIA A100-SXM4-40GB in Colab, Tokens/s
A100_TOKENS_PER_SECOND = {
    "1 Ausgangsstand": 13_760,
    "2 16 Bit mit autocast": 35_447,
    "3 16 Bit, ganzes Modell": 50_702,
    "4 torch.nn ohne Flash": 50_457,
    "5 torch.nn mit Flash": 75_621,
    "6 torch.compile": 92_586,
}

if not BENCHMARKS:
    print("Noch kein Benchmark. Job 1 auf einem Knoten misst ihn: sbatch slurm/check.sbatch")
for record in BENCHMARKS:
    gpu = record["environment"]["gpu"] or "CPU"
    variants = pd.DataFrame(record["variants"]).set_index("variant")
    table = pd.DataFrame(
        {
            "A100 Tokens/s (Woche 5)": pd.Series(A100_TOKENS_PER_SECOND),
            f"{gpu} Tokens/s": variants["tokens_per_second"],
            "Speicher (GB)": variants["peak_memory_gb"],
            "MFU": variants["mfu"],
            "Fehler": variants["error"],
        }
    )
    table[f"{gpu} gegenüber A100"] = table[f"{gpu} Tokens/s"] / table["A100 Tokens/s (Woche 5)"]
    print(Path(record["file"]).name)
    display(table)
    scan = pd.DataFrame(record["batch_scan"]).set_index("batch_size")
    display(scan[["tokens_per_second", "peak_memory_gb", "mfu", "error"]])
    fits = scan.index[scan["error"].isna()]
    print("Größte Batchgröße ohne Speicherfehler:", fits.max() if len(fits) else "keine")

## 3. Woche 4 gegen optimiert (Lauf Nr. 2 und 3)

Erwartet, vor dem Messen festgehalten: GPT-2 original kommt auf einen Test-Loss von 3,63 wie in Colab, von Grund auf ungefähr 4,35. Nr. 3 sollte einen ähnlichen Test-Loss haben, aber mehrfach so viele Tokens/s.

In [ ]:
week4, optimized = by_group("week6-week4"), by_group("week6-optimized")
if not week4 and not optimized:
    print("Noch keine Läufe Nr. 2 und 3. Start: sbatch slurm/train.sbatch --config configs/lotr-small-week4.toml")
else:
    display(summary(week4 + optimized))
    for record in week4:
        if record["pretrained"] is not None:
            pretrained = record["pretrained"]
            print(f"GPT-2 original: Test-Loss {pretrained['test_loss']:.3f}, Perplexity {pretrained['perplexity']:.1f}")
    speeds = [(record["result"] or {}).get("tokens_per_second") for record in (week4[:1] + optimized[:1])]
    if len(speeds) == 2 and None not in speeds:
        print(f"Optimiert gegenüber Woche 4: {speeds[1] / speeds[0]:.1f}-mal so viele Tokens/s")

## 4. Suche nach Lernrate und Batchgröße (Lauf Nr. 4)

Jede Kombination trainiert eine Epoche. Bei Batch 8 sind das gut 260 Schritte, bei Batch 128 nur 16. Kleine Batches gewinnen deshalb nach einer Epoche fast sicher, das ist der Zielkonflikt von Folie 26. Ein gescheiterter Lauf steht mit seinem Fehler in der Tabelle, ein fehlender mit „fehlt“.

In [ ]:
from infm_sota2.cluster.results import search_table

records = by_group("week6-search")
if not records:
    print("Noch keine Läufe der Suche. Start: sbatch slurm/search.sbatch")
else:
    rows = pd.DataFrame(search_table(records, SEARCH.learning_rates, SEARCH.batch_sizes))
    print("Bester Validierungs-Loss, Zeilen Lernrate, Spalten Batchgröße:")
    display(rows.pivot(index="learning_rate", columns="batch_size", values="best_val_loss"))
    print("Status:")
    display(rows.pivot(index="learning_rate", columns="batch_size", values="status"))
    ok = rows[rows["status"] == "ok"]
    if len(ok):
        best = ok.loc[ok["best_val_loss"].idxmin()]
        print(f"Beste Kombination: Lernrate {best.learning_rate:g}, Batch {best.batch_size}, "
              f"Validierungs-Loss {best.best_val_loss:.3f}. In lotr-small-main.toml eintragen.")
        middle = ok[ok["batch_size"] == sorted(SEARCH.batch_sizes)[len(SEARCH.batch_sizes) // 2]]
        if len(middle):
            lr = middle.loc[middle["best_val_loss"].idxmin()].learning_rate
            print(f"Beste Lernrate für Batch {middle.batch_size.iloc[0]}: {lr:g}. In lotr-small-scaling.toml eintragen.")

## 5. Skalierung (Lauf Nr. 5)

Dieselbe Gesamt-Batchgröße 32 auf 1, 2 und 4 Knoten, die Arbeit pro Schritt bleibt also gleich. Speedup S_p = Tokens/s mit p GPUs geteilt durch Tokens/s mit einer, Effizienz E_p = S_p / p (Folie 5). Erwartet: Der Speedup bleibt deutlich unter p. Pro Schritt gehen rund 650 MB Gradienten über InfiniBand, das ist der serielle Anteil aus Folie 6. Fast gleiche Losses bei allen Knotenzahlen zeigen, dass DDP richtig rechnet.

In [ ]:
from infm_sota2.cluster.results import scaling_table

records = by_group("week6-scaling")
baseline = [r for r in records if r["environment"]["world_size"] == 1 and r["error"] is None]
if not records:
    print("Noch keine Läufe der Skalierung. Start: sbatch --nodes=1 slurm/train.sbatch --config configs/lotr-small-scaling.toml")
elif not baseline:
    print("Für Speedup und Effizienz fehlt noch ein erfolgreicher Lauf mit einer GPU (--nodes=1).")
else:
    rows = pd.DataFrame(scaling_table(records))
    losses = {r["environment"]["world_size"]: (r["result"] or {}).get("best_val_loss") for r in records}
    rows["bester Val-Loss"] = rows["world_size"].map(losses)
    display(rows.set_index("world_size"))
    ok = rows[rows["status"] == "ok"]
    gpus = list(rows["world_size"])
    fig, (ax_speedup, ax_efficiency) = plt.subplots(1, 2, figsize=(10, 4))
    ax_speedup.plot(gpus, gpus, linestyle="--", color=MUTED)
    ax_speedup.annotate("ideal", (gpus[-1], gpus[-1]), textcoords="offset points", xytext=(-8, 6), ha="right", color=MUTED)
    ax_speedup.plot(ok["world_size"], ok["speedup"], marker="o", color=SERIES[0])
    ax_efficiency.axhline(1.0, linestyle="--", color=MUTED)
    ax_efficiency.plot(ok["world_size"], ok["efficiency"], marker="o", color=SERIES[0])
    for ax, column, fmt in ((ax_speedup, "speedup", "{:.2f}"), (ax_efficiency, "efficiency", "{:.0%}")):
        for x, y in zip(ok["world_size"], ok[column]):
            if x > 1:  # die Basis ist per Definition 1
                ax.annotate(fmt.format(y), (x, y), textcoords="offset points", xytext=(0, -16), ha="center", color=INK_SECONDARY)
        ax.set_xticks(gpus)
        ax.set_xlabel("GPUs (Knoten mit je einer L40S)")
    ax_speedup.set_title("Speedup S_p")
    ax_efficiency.set_title("Effizienz E_p = S_p / p")
    ax_speedup.set_ylim(0, max(gpus) * 1.15)
    ax_efficiency.set_ylim(0, 1.15)
    plt.tight_layout()
    plt.show()

## 6. Hauptlauf, Medium und Large (Lauf Nr. 6 bis 8)

20 Epochen mit der besten Kombination der Suche, gespeichert wird der Stand mit dem besten Validierungs-Loss. Der Verlauf zeigt, ab wann das Modell anfängt auswendig zu lernen: Dann steigt der Validierungs-Loss wieder. Der Punkt auf jeder Kurve markiert den besten Stand. Zum Vergleich: In Woche 4 kam GPT-2 Small von Grund auf nach 4 Epochen auf einen Test-Loss von ungefähr 4,35.

In [ ]:
runs = by_group("week6-main") + by_group("week6-medium") + by_group("week6-large")
if not runs:
    print("Noch keine Läufe Nr. 6 bis 8. Start: sbatch --nodes=2 slurm/train.sbatch --config configs/lotr-small-main.toml")
else:
    display(summary(runs))
    curves = [r for r in runs if (r["result"] or {}).get("val_losses")]
    if len(curves) > len(SERIES):
        print(f"{len(curves)} Läufe mit Verlauf, das Diagramm zeigt die ersten {len(SERIES)}. PERSON grenzt ein.")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ends = []
    for color, record in zip(SERIES, curves):
        steps, values = zip(*record["result"]["val_losses"])
        train = record["config"]["train"]
        # Tokens statt Schritte, damit Läufe mit verschiedener Batchgröße vergleichbar sind
        tokens = [step * train["batch_size"] * record["config"]["data"]["seq_len"] / 1e6 for step in steps]
        ax.plot(tokens, values, color=color, label=record["label"])
        best = values.index(min(values))
        ax.plot(tokens[best], values[best], marker="o", color=color)
        ends.append((values[-1], tokens[-1], record["label"]))
    # Beschriftung am Kurvenende, nahe Enden mit Mindestabstand übereinander
    low, high = ax.get_ylim()
    gap, previous = 0.05 * (high - low), None
    for y, x, label in sorted(ends):
        y = y if previous is None else max(y, previous + gap)
        ax.annotate(label, (x, y), textcoords="offset points", xytext=(6, 0), va="center", color=INK_SECONDARY)
        previous = y
    ax.set_xlim(right=max(x for _, x, _ in ends) * 1.35)  # Platz für die Beschriftungen
    ax.set_xlabel("Tokens im Training (Mio.)")
    ax.set_ylabel("Validierungs-Loss")
    ax.set_title("Validierungs-Loss im Verlauf")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()
    for record in runs:
        show_texts(record)

## 7. Fazit

Zum Ausfüllen, sobald die Läufe fertig sind:

- **Cluster gegen Colab (Nr. 2):** Stimmt der Test-Loss von GPT-2 original mit 3,63 überein? Wie nah kommt der Lauf von Grund auf an 4,35?
- **Optimierungen (Nr. 3):** Wie viel schneller ist das echte Training, und bleibt der Loss gleich?
- **Suche (Nr. 4):** Welche Kombination gewinnt, und wie stark liegt das an der Zahl der Schritte pro Epoche?
- **Skalierung (Nr. 5):** Speedup und Effizienz bei 2 und 4 Knoten, und was davon erklärt die Übertragung der Gradienten?
- **Hauptlauf und Medium (Nr. 6 und 7):** Test-Loss gegenüber Woche 4, ab welcher Epoche das Modell auswendig lernt, und was das größere Modell bringt.